# Name That LLM | Lean two-stage Gaussian ensemble

Hypothesis: the main SVC + 8B/14B two-stage LoRA branch may retain enough quality to justify removing the complementary 15% ensemble. Skip that ensemble entirely and renormalize the remaining branch to weight 1.0. Keep average-linkage groups, LoRA weights, Gaussian temperatures, Sinkhorn and punctuation/Hungarian rules from iteration 07.

The user reports 0.604 and roughly four hours for iteration 07. This ablation is not locally selected or scored; quality may decrease. Removing a whole multi-model branch reduces the work performed, but the runtime reduction has not been measured.

# Credits and provenance

Adapted from **Sangam Gautam (sangam21)**: [Two-stage LoRA + Gaussian + full Hungarian (reported 0.604)](https://www.kaggle.com/code/sangam21/two-stage-lora-gaussian-full-hungarian-0-604). The title score is an author-reported reference; we have not independently reproduced it.

Upstream components include [Nishant Dahal's two-stage LoRA](https://www.kaggle.com/code/nishantdahal/two-stage-lora-on-the-public-pool), [prvsiyan's shared-prompt Gaussian work](https://www.kaggle.com/prvsiyan), and [Lucas Boesen's public models and ensembles](https://www.kaggle.com/lucasboesen). Additional checkpoint and dataset authors are credited by the linked source and the input list below, including Vladislav Lavrentev and Zbigniew Kirchner. Retain each upstream asset's license and attribution; this adaptation does not relicense them.


In [ ]:
'''1. Environment
Use GPU T4 x2, Internet off, and the same package environment/checkpoint revisions as iteration 07.
BLAS operations are limited to four CPU threads to reduce oversubscription; this is not a measured speedup.'''

In [ ]:
import tempfile
import ast
import gc
import glob
import hashlib
import itertools
import json
import os
from pathlib import Path
import re
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.optimize import linear_sum_assignment
from scipy.sparse import hstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from contextlib import contextmanager
from threadpoolctl import threadpool_limits

In [ ]:
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
LORAS = {'lora8b_v1': dict(w=0.226, twin_w=0.079, bs=16, base=dict(model_type='qwen3', hidden_size=4096, num_hidden_layers=36)), 'lora14b_v1': dict(w=0.429, twin_w=0.15, bs=8, base=dict(model_type='qwen3', hidden_size=5120, num_hidden_layers=40))}
W_SVC = 1.021
FUNCTION_WORDS = set('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char', 3, 5, 250000, 1.0), ('word', 1, 3, 180000, 0.7), ('shape', 2, 6, 60000, 0.5), ('function', 1, 3, 50000, 0.5)]
SOURCE_URL = 'https://www.kaggle.com/code/sangam21/two-stage-lora-gaussian-full-hungarian-0-604'
SOURCE_SHA256 = 'ddfa6f2f4f2774ecade7e58935238968cc4a3db0ddf760591e3751cc87aa5613'
GAUSSIAN_SOURCE_SHA256 = 'b3388879ecf798ad32e42b18b8be3a3157a6ed252e53df7a6c948b8374202d76'

In [ ]:
'''2. Required inputs
Only the competition CSVs, pool_v1.parquet, lora8b_v1/lora14b_v1 checkpoints and matching Qwen3 8B/14B bases are used. Other inherited attachments are unused.'''

In [ ]:
SOURCE_INPUTS = {'competition_sources': ['name-that-llm'], 'dataset_sources': ['nishantdahal/ntl-two-stage-lora'], 'kernel_sources': [], 'model_sources': ['qwen-lm/qwen-3/Transformers/14b-base/1', 'qwen-lm/qwen-3/Transformers/8b-base/1'], 'docker_image': 'gcr.io/kaggle-private-byod/python@sha256:2757e0c7d1e0a9cb43da657b97e223c321a98f5014bdf64f44f2f6b083ad2b2f'}
print(json.dumps(SOURCE_INPUTS, indent=2))

In [ ]:
'''3. Configuration, input validation and stage timing'''

In [ ]:
class ExperimentConfig:
    def __init__(self):
        self.input_root = Path('/kaggle/input')
        self.output_root = Path('/kaggle/working')
        self.mode = 'lean_two_stage'
        self.use_complement = False
        self.linkage_method = 'average'
        self.cluster_threshold = 0.85
        self.labels = list(LABELS)
        self.loras = LORAS


In [ ]:
class CompetitionAssets:
    def __init__(self, config):
        self.config = config

    def find(self, pattern):
        matches = sorted(glob.glob(f'{self.config.input_root}/**/{pattern}', recursive=True))
        if not matches:
            raise FileNotFoundError(f'Missing required Kaggle input: {pattern}')
        return matches[0]

    def load(self):
        directory = next((p for p in [self.config.input_root / 'competitions/name-that-llm',
                                      self.config.input_root / 'name-that-llm']
                          if (p / 'train.csv').is_file()), None)
        if directory is None:
            raise FileNotFoundError('Attach the Name That LLM competition data')
        train = pd.read_csv(directory / 'train.csv')
        test = pd.read_csv(directory / 'test.csv')
        if not {'id', 'text', 'model', 'prompt_id'}.issubset(train) or not {'id', 'text'}.issubset(test):
            raise ValueError('Unexpected competition data schema')
        for frame in (train, test):
            if frame.empty or frame['id'].isna().any() or frame['id'].duplicated().any():
                raise ValueError('Expected nonempty data with unique, non-null IDs')
        if train['model'].isna().any() or set(train['model']) != set(self.config.labels):
            raise ValueError('The training set must contain all 18 known author labels')
        test['text'] = test['text'].fillna('').astype(str)
        targets = train['model'].map({label: i for i, label in enumerate(self.config.labels)}).to_numpy()
        return train, test, targets

    def preflight(self):
        if os.environ.get('NTL_SMOKE') == '1':
            raise RuntimeError('Disable NTL_SMOKE for a real competition run')
        if torch.cuda.device_count() != 2 or any('T4' not in torch.cuda.get_device_name(i) for i in range(2)):
            raise RuntimeError('Select GPU T4 x2 and attach the original notebook inputs')
        markers = ['pool_v1.parquet', 'lora8b_v1/meta.json', 'lora14b_v1/meta.json']
        for marker in markers:
            self.find(marker)

In [ ]:
class StageTimer:
    def __init__(self, output_root):
        self.path = output_root / 'stage_timings.json'
        self.records = []

    def save(self):
        temporary = self.path.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(self.records, indent=2))
        temporary.replace(self.path)

    @contextmanager
    def measure(self, name):
        record = {'stage': name, 'status': 'running'}
        self.records.append(record)
        self.save()
        print(f'START {name}', flush=True)
        started = time.monotonic()
        try:
            yield
        except BaseException:
            record['status'] = 'failed'
            raise
        else:
            record['status'] = 'complete'
        finally:
            record['seconds'] = time.monotonic() - started
            self.save()
            print(f"END {name}: {record['status']}, {record['seconds'] / 60:.1f} min", flush=True)


In [ ]:
'''4. Style classifier'''

In [ ]:
class StyleClassifier:
    def __init__(self, labels):
        self.labels = labels

    @staticmethod
    def shape_text(text):
        return re.sub(r'[^\W\d_]+', lambda match: 'X' if match[0].isupper() else 'x', text)

    @staticmethod
    def function_text(text):
        return ' '.join(word.lower() if word.lower() in FUNCTION_WORDS or not word.isalnum() else 'CONTENT'
                        for word in re.findall(r'\w+|[^\w\s]', text))

    def fit_predict(self, train, test, targets, pool):
        extra_targets = pool['model'].map({label: i for i, label in enumerate(self.labels)})
        if extra_targets.isna().any() or pool['text'].isna().any():
            raise ValueError('The public pool contains invalid labels or text')
        texts = train['text'].astype(str).tolist() + pool['text'].tolist()
        fit_targets = np.r_[targets, extra_targets.to_numpy()]
        training, query = [], []
        for view, low, high, max_features, weight in VIEWS:
            prepare = self.shape_text if view == 'shape' else self.function_text if view == 'function' else str
            analyzer = 'char' if view == 'shape' else 'word' if view == 'function' else view
            options = dict(analyzer=analyzer, ngram_range=(low, high), lowercase=False,
                           sublinear_tf=True, min_df=2, max_features=max_features, dtype=np.float32)
            if analyzer == 'word':
                options['token_pattern'] = r'(?u)\b\w+\b|[^\w\s]'
            vectorizer = TfidfVectorizer(**options)
            training.append(vectorizer.fit_transform([prepare(text) for text in texts]) * weight)
            query.append(vectorizer.transform([prepare(text) for text in test['text']]) * weight)
        classifier = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced')
        classifier.fit(hstack(training, format='csr'), fit_targets)
        if not np.array_equal(classifier.classes_, np.arange(len(self.labels))):
            raise ValueError('Unexpected class order in the style classifier')
        return ProbabilityMath.softmax(classifier.decision_function(hstack(query, format='csr')))

In [ ]:
'''5. Probability calibration'''

In [ ]:
class ProbabilityMath:
    @staticmethod
    def softmax(values):
        exponential = np.exp(values - values.max(1, keepdims=True))
        return exponential / exponential.sum(1, keepdims=True)

    @staticmethod
    def balance(values):
        probabilities = ProbabilityMath.softmax(values)
        if len(probabilities) >= 50:
            target = np.full(probabilities.shape[1], len(probabilities) / probabilities.shape[1])
            for _ in range(100):
                probabilities *= target / probabilities.sum(0)
                probabilities /= probabilities.sum(1, keepdims=True)
        return probabilities

    @staticmethod
    def punctuation_guard(probabilities, mask, labels):
        result = probabilities.copy()
        allowed = {'claude-opus-4-6', 'claude-opus-5', 'glm-5.3', 'glm-5.3-flash'}
        forbidden = np.array([label not in allowed for label in labels])
        for row in np.flatnonzero(mask):
            if labels[result[row].argmax()] not in allowed:
                result[row, forbidden] = 0
                mass = result[row].sum()
                if mass <= 0:
                    raise ValueError('The inherited punctuation guard removed all probability mass')
                result[row] /= mass
        return result

In [ ]:
'''6. Shared-prompt Gaussian core; credited to prvsiyan'''

In [ ]:
class SharedPromptGaussian:
    K = 18
    DEFAULT = {'red': 'wpca', 'd': 256, 'se_shrink': 'lw', 'sp_shrink': 'lw', 'lam_floor': 0.0001, 'n_exact': 3, 'inj': True, 'iters': 5, 'repel': True, 'sp_scale': 1.0}
    _MASK = {}
    LAM_DIFF = 1000000.0

    @staticmethod
    def _lse(A, axis=None, keepdims=False):
        m = np.max(A, axis=axis, keepdims=True)
        m = np.where(np.isfinite(m), m, 0.0)
        out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
        if keepdims:
            return out
        return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

    @staticmethod
    def log_softmax(L):
        return L - SharedPromptGaussian._lse(L, axis=1, keepdims=True)

    @staticmethod
    def _lw_shrinkage(X):
        '''Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered).'''
        from sklearn.covariance import ledoit_wolf_shrinkage
        return float(ledoit_wolf_shrinkage(X, assume_centered=True))

    @staticmethod
    def _shrunk(S, a):
        d = S.shape[0]
        return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

    @staticmethod
    def _two_way(Z, y, pinv, G, iters=4):
        '''additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds).'''
        P = np.zeros((G, Z.shape[1]))
        cnt_p = np.bincount(pinv, minlength=G).astype(float)
        cls = np.unique(y)
        mu = np.zeros((SharedPromptGaussian.K, Z.shape[1]))
        for _ in range(iters):
            R = Z - P[pinv]
            for c in cls:
                mu[c] = R[y == c].mean(0)
            R = Z - mu[y]
            P = np.zeros_like(P)
            np.add.at(P, pinv, R)
            P /= cnt_p[:, None]
        return (mu, P, cnt_p)

    @staticmethod
    def fit(X, y, prompts, cfg=None):
        '''fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit).'''
        c = dict(SharedPromptGaussian.DEFAULT)
        c.update(cfg or {})
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        N, D = X.shape
        pu, pinv = np.unique(np.asarray(prompts), return_inverse=True)
        G = len(pu)
        mx = X.mean(0)
        sx = X.std(0)
        sx = np.where(sx > 1e-08, sx, 1.0)
        Xs = (X - mx) / sx
        if c['red'] == 'none':
            V = np.eye(D)
        else:
            if c['red'] == 'wpca':
                Pm = np.zeros((G, D))
                np.add.at(Pm, pinv, Xs)
                Pm /= np.bincount(pinv)[:, None]
                R = Xs - Pm[pinv]
            elif c['red'] == 'pca':
                R = Xs
            else:
                raise ValueError(c['red'])
            d = int(min(c['d'], N - G - 1, D))
            if N < D:
                w, U = np.linalg.eigh(R @ R.T)
                o = np.argsort(w)[::-1][:d]
                w, U = (np.clip(w[o], 1e-12, None), U[:, o])
                V = R.T @ U / np.sqrt(w)
            else:
                _, _, Vt = np.linalg.svd(R, full_matrices=False)
                V = Vt[:d].T
        Z = Xs @ V
        d = Z.shape[1]
        mu, P, ng = SharedPromptGaussian._two_way(Z, y, pinv, G)
        E = Z - mu[y] - P[pinv]
        dof = max(N - G - len(np.unique(y)) + 1, 1)
        Se_raw = E.T @ E / dof
        ae = SharedPromptGaussian._lw_shrinkage(E) if c['se_shrink'] == 'lw' else float(c['se_shrink'])
        Se = SharedPromptGaussian._shrunk(Se_raw, ae)
        Pc = P - P.mean(0)
        Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
        ap = SharedPromptGaussian._lw_shrinkage(Pc) if c['sp_shrink'] == 'lw' else float(c['sp_shrink'])
        tr = max(np.trace(Sp_raw), 1e-08 * np.trace(Se))
        Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)
        Sp *= c['sp_scale']
        L = np.linalg.cholesky(Se)
        Li = np.linalg.inv(L)
        M = Li @ Sp @ Li.T
        M = 0.5 * (M + M.T)
        lam, Q = np.linalg.eigh(M)
        lam = np.clip(lam, c['lam_floor'], None)
        A = Q.T @ Li
        W = V @ A.T
        return {'mx': mx, 'sx': sx, 'W': W, 'm': mu @ A.T, 'lam': lam, 'T1': 1.0, 'Tg': 1.0, 'info': {'d': int(d), 'a_e': round(ae, 4), 'a_p': round(ap, 4), 'lam_pct': np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, 'cfg': c}

    @staticmethod
    def transform(X, prm):
        return (np.asarray(X, dtype=np.float64) - prm['mx']) / prm['sx'] @ prm['W']

    @staticmethod
    def _inj_mask(n, C):
        if (n, C) not in SharedPromptGaussian._MASK:
            grids = np.meshgrid(*[np.arange(C)] * n, indexing='ij')
            bad = np.zeros([C] * n, bool)
            for i, j in itertools.combinations(range(n), 2):
                bad |= grids[i] == grids[j]
            SharedPromptGaussian._MASK[n, C] = bad
        return SharedPromptGaussian._MASK[n, C]

    @staticmethod
    def singleton_ll(U, m, lam):
        '''(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out).'''
        R = U[:, None, :] - m[None]
        return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

    @staticmethod
    def joint_terms(U, m, lam):
        '''a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped).'''
        n = len(U)
        gam = lam / (1 + n * lam)
        R = U[:, None, :] - m[None]
        a = -0.5 * (R ** 2 * (1 - gam)).sum(-1)
        G = R * np.sqrt(gam)
        B = np.einsum('icd,jed->ijce', G, G)
        return (a, B)

    @staticmethod
    def joint_table(a, B):
        '''full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij.'''
        n, C = a.shape
        Q = np.zeros([C] * n)
        for i in range(n):
            sh = [1] * n
            sh[i] = C
            Q = Q + a[i].reshape(sh)
        for i, j in itertools.combinations(range(n), 2):
            sh = [1] * n
            sh[i] = C
            sh[j] = C
            Q = Q + B[i, j].reshape(sh)
        return Q

    @staticmethod
    def exact_marginals(Q, T=1.0, inj=True):
        '''per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T.'''
        n = Q.ndim
        C = Q.shape[0]
        Qt = Q / T
        if inj and n >= 2:
            Qt = np.where(SharedPromptGaussian._inj_mask(n, C), -np.inf, Qt)
        lz = SharedPromptGaussian._lse(Qt)
        out = np.empty((n, C))
        for i in range(n):
            ax = tuple((k for k in range(n) if k != i))
            out[i] = (SharedPromptGaussian._lse(Qt, axis=ax) if ax else Qt) - lz
        return out

    @staticmethod
    def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
        '''cavity EM for one group (any n >= 2): returns (n, C) log q.'''
        n = len(U)
        v = lam / (1 + (n - 1) * lam)
        L = SharedPromptGaussian.singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
        q = np.exp(SharedPromptGaussian.log_softmax(L))
        for _ in range(iters):
            s = U - q @ m
            S = s.sum(0)
            Pm = (S[None] - s) * v
            R = U[:, None, :] - Pm[:, None, :] - m[None]
            L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
            if repel:
                lo = np.log(np.clip(1 - q, 1e-06, None))
                L = L + (lo.sum(0)[None] - lo)
            q = np.exp(SharedPromptGaussian.log_softmax(L))
        return SharedPromptGaussian.log_softmax(L)

    @staticmethod
    def lam_g(prm, cfg=None):
        '''lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly.'''
        c = dict(prm.get('cfg', SharedPromptGaussian.DEFAULT))
        c.update(cfg or {})
        return np.full_like(prm['lam'], SharedPromptGaussian.LAM_DIFF) if c.get('diff') else prm['lam']

    @staticmethod
    def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
        c = dict(prm.get('cfg', SharedPromptGaussian.DEFAULT))
        c.update(cfg or {})
        n = len(U)
        lg = SharedPromptGaussian.lam_g(prm, cfg)
        T1 = prm['T1'] if T1 is None else T1
        Tg = prm['Tg'] if Tg is None else Tg
        if n == 1:
            return SharedPromptGaussian.log_softmax(SharedPromptGaussian.singleton_ll(U, prm['m'], prm['lam']) / T1)
        if n <= c['n_exact']:
            a, B = SharedPromptGaussian.joint_terms(U, prm['m'], lg)
            return SharedPromptGaussian.exact_marginals(SharedPromptGaussian.joint_table(a, B), Tg, c['inj'])
        return SharedPromptGaussian.em_marginals(U, prm['m'], lg, Tg, c['iters'], c['repel'])

    @staticmethod
    def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
        '''(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed).'''
        U = SharedPromptGaussian.transform(X_pool, prm) if U is None else U
        lab = np.asarray(lab)
        n = len(U)
        out = np.empty((n, prm['m'].shape[0]))
        sizes = np.bincount(lab)
        s1 = sizes[lab] == 1
        T1 = prm['T1'] if T1 is None else T1
        if s1.any():
            out[s1] = SharedPromptGaussian.log_softmax(SharedPromptGaussian.singleton_ll(U[s1], prm['m'], prm['lam']) / T1)
        for g in np.where(sizes >= 2)[0]:
            ii = np.where(lab == g)[0]
            out[ii] = SharedPromptGaussian.group_logpost(U[ii], prm, T1, Tg, cfg)
        return out

In [ ]:
'''7. Original two-stage LoRA pooling with progress logging'''

In [ ]:
class CheckpointResolver:
    @staticmethod
    def has_weights(directory):
        directory = Path(directory)
        if any((directory / name).is_file() for name in ('model.safetensors', 'pytorch_model.bin')):
            return True
        for name in ('model.safetensors.index.json', 'pytorch_model.bin.index.json'):
            try:
                shards = set(json.loads((directory / name).read_text())['weight_map'].values())
                if shards and all((directory / shard).is_file() for shard in shards):
                    return True
            except (OSError, ValueError, KeyError, TypeError, AttributeError):
                continue
        return False

    @staticmethod
    def find(input_root, spec=None, dir_name=None):
        root = Path(input_root)
        pattern = f'**/{dir_name}/config.json' if dir_name else '**/config.json'
        candidates = sorted(root.glob(pattern))
        candidates.sort(key=lambda path: 0 if root / 'models' in path.parents else 1)
        for config_path in candidates:
            try:
                config = json.loads(config_path.read_text())
            except (OSError, ValueError):
                continue
            if not isinstance(config, dict):
                continue
            if (dir_name or all(config.get(key) == value for key, value in (spec or {}).items())):
                if CheckpointResolver.has_weights(config_path.parent):
                    print(f'Checkpoint: {config_path.parent}', flush=True)
                    return str(config_path.parent)
        raise FileNotFoundError(f'No checkpoint with complete local weights: {dir_name or spec} under {root}')

    @staticmethod
    def compile_stack(source):
        tree = ast.parse(source)
        functions = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == 'find_checkpoint']
        if len(functions) != 1 or [arg.arg for arg in functions[0].args.args] != ['model_type', 'hidden', 'dir_name']:
            raise ValueError('Unexpected upstream checkpoint resolver; review the compatibility patch')
        functions[0].body = ast.parse(
            "spec = {'model_type': model_type}\n"
            "if hidden is not None: spec['hidden_size'] = hidden\n"
            "return CheckpointResolver.find('/kaggle/input', spec, dir_name)"
        ).body
        return compile(ast.fix_missing_locations(tree), 'credited_public_gaussian_stack', 'exec')


In [ ]:
class CheckpointResolverChecks:
    @staticmethod
    def run():
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            config_only = root / 'a_archive/base_config'
            weighted = root / 'z_base'
            config_only.mkdir(parents=True)
            weighted.mkdir()
            spec = {'model_type': 'qwen3', 'hidden_size': 4096}
            for directory in (config_only, weighted):
                (directory / 'config.json').write_text(json.dumps(spec))
            (weighted / 'model.safetensors.index.json').write_text(json.dumps(
                {'weight_map': {'layer0': 'model-00001-of-00002.safetensors',
                                'layer1': 'model-00002-of-00002.safetensors'}}))
            (weighted / 'model-00001-of-00002.safetensors').write_bytes(b'fixture')
            assert not CheckpointResolver.has_weights(config_only)
            assert not CheckpointResolver.has_weights(weighted)
            (weighted / 'model-00002-of-00002.safetensors').write_bytes(b'fixture')
            assert CheckpointResolver.find(root, spec) == str(weighted)
            assert CheckpointResolver.find(root, dir_name='z_base') == str(weighted)
            (weighted / 'model-00002-of-00002.safetensors').unlink()
            try:
                CheckpointResolver.find(root, spec)
            except FileNotFoundError:
                pass
            else:
                raise AssertionError('A checkpoint with missing shards was selected')

CheckpointResolverChecks.run()


In [ ]:
class LoRAFeatureExtractor:

    def __init__(self, input_dir, labels):
        self.input_dir = input_dir
        self.labels = labels

    def find_base(self, spec):
        return CheckpointResolver.find(self.input_dir, spec)

    def load_trunk(self, base_dir, keep, dtype):
        '''First `keep` decoder layers, split over both T4s when it does not fit on one.'''
        mc = AutoConfig.from_pretrained(base_dir)
        mc.num_hidden_layers, mc.use_cache = (keep, False)
        if getattr(mc, 'layer_types', None) is not None:
            mc.layer_types = mc.layer_types[:keep]
        hd = getattr(mc, 'head_dim', mc.hidden_size // mc.num_attention_heads)
        gib = 2 * (mc.vocab_size * mc.hidden_size + keep * (2 * mc.hidden_size * mc.num_attention_heads * hd + 2 * mc.hidden_size * mc.num_key_value_heads * hd + 3 * mc.hidden_size * mc.intermediate_size)) / 2 ** 30
        if gib > 13 and torch.cuda.device_count() > 1:
            mem = {i: '13GiB' for i in range(torch.cuda.device_count())}
            model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map='auto', max_memory=mem).eval()
        else:
            model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to('cuda:0').eval()
        model.norm = nn.Identity()
        return model

    @torch.no_grad()
    def merge_lora(self, model, adapter_dir):
        acfg = json.load(open(os.path.join(adapter_dir, 'adapter_config.json')))
        scale = acfg['lora_alpha'] / acfg['r']
        sd = load_file(os.path.join(adapter_dir, 'adapter_model.safetensors'))
        modules = dict(model.named_modules())
        for k, A in sd.items():
            if '.lora_A.' in k:
                name = k.split('.lora_A.')[0]
                lin = modules[name[name.index('layers.'):]]
                Bm = sd[k.replace('.lora_A.', '.lora_B.')]
                lin.weight.add_((Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device) * scale).to(lin.weight.dtype))

    @torch.no_grad()
    def lora_features(self, ckpt_dir, base_dir, texts, bs):
        '''Pooled features Z (n, 8 * hidden) and the checkpoint's heads.'''
        meta = json.load(open(os.path.join(ckpt_dir, 'meta.json')))
        assert meta['labels'] == self.labels, "label order differs from the checkpoint's heads"
        cfg = meta['cfg']
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        tok = AutoTokenizer.from_pretrained(base_dir)
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        model = self.load_trunk(base_dir, cfg['keep'], dtype)
        self.merge_lora(model, os.path.join(ckpt_dir, 'adapter'))
        heads = load_file(os.path.join(ckpt_dir, 'heads.safetensors'))
        layers, H = (cfg['layers'], model.config.hidden_size)
        norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to('cuda:0').float()
        norms.load_state_dict({k[len('norms.'):]: v for k, v in heads.items() if k.startswith('norms.')})
        ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg['max_len'])['input_ids']
        ids = [x + [tok.eos_token_id] for x in ids]
        order = np.argsort([len(x) for x in ids], kind='stable')
        Z = np.zeros((len(ids), H * len(layers)), np.float32)
        progress_started = time.monotonic()
        progress_every = max(bs, ((len(order) + 19 * bs) // (20 * bs)) * bs)
        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            enc = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(model.embed_tokens.weight.device)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            if not all((torch.isfinite(hs[l]).all() for l in layers)):
                model.to(torch.bfloat16)
                hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
                model.to(dtype)
            m = enc['attention_mask'].unsqueeze(-1).float().to('cuda:0')
            m[:, 0] = 0
            den = m.sum(1).clamp(min=1)
            Z[idx] = torch.cat([nm((hs[l].to('cuda:0').float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()
            if s % progress_every == 0 or s + bs >= len(order):
                done = min(s + bs, len(order))
                elapsed = time.monotonic() - progress_started
                print(f'LoRA passages {done}/{len(order)}; elapsed {elapsed / 60:.1f} min', flush=True)
            del hs, enc, m, den
        del model
        torch.cuda.empty_cache()
        return (Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith('norms.')})

In [ ]:
'''8. Topic grouping and decoding'''

In [ ]:
class TopicGrouping:
    def __init__(self, threshold=0.85, method='average'):
        if method not in {'average', 'complete'}:
            raise ValueError('Expected average or complete linkage')
        self.threshold = threshold
        self.method = method

    def fit_predict(self, texts):
        texts = list(texts)
        if not texts:
            raise ValueError('Cannot group an empty test pool')
        features = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(texts)
        distance = np.clip(1 - (features @ features.T).toarray(), 0, None)
        np.fill_diagonal(distance, 0)
        groups = (fcluster(linkage(squareform(distance, checks=False), method=self.method),
                           self.threshold, criterion='distance') if len(texts) >= 2
                  else np.ones(len(texts), dtype=int))
        sizes = np.bincount(groups)[groups]
        maximum = np.zeros(len(groups), dtype=float)
        for group in np.unique(groups):
            rows = np.flatnonzero(groups == group)
            if 2 <= len(rows) <= 18:
                maximum[rows] = distance[np.ix_(rows, rows)].max()
                if self.method == 'complete':
                    assert maximum[rows].max() <= self.threshold + 1e-12
        return groups, (sizes >= 2) & (sizes <= 18), maximum


In [ ]:
class HungarianDecoder:
    def __init__(self, guarded=False, threshold=0.85):
        self.guarded = guarded
        self.threshold = threshold
        self.audit = []

    def decode(self, probabilities, groups, max_distances):
        if not np.isfinite(probabilities).all() or (probabilities < 0).any():
            raise ValueError('Hungarian decoding requires finite, nonnegative probabilities')
        result = probabilities.copy()
        self.audit = []
        for group in np.unique(groups):
            members = np.flatnonzero(groups == group)
            if not 12 <= len(members) <= 18:
                continue
            accepted = not self.guarded or max_distances[members].max() <= self.threshold + 1e-12
            record = {'group': int(group), 'rows': len(members), 'accepted': bool(accepted),
                      'max_pair_distance': float(max_distances[members].max()), 'changed': 0}
            if accepted:
                row_index, column_index = linear_sum_assignment(-np.log(np.clip(result[members], 1e-12, 1)))
                assigned = members[row_index]
                record['changed'] = int((result[assigned].argmax(1) != column_index).sum())
                result[assigned] = 0
                result[assigned, column_index] = 1
            self.audit.append(record)
        return result

In [ ]:
class GroupingContractChecks:
    @staticmethod
    def run():
        distances = np.array([[0., 0.4, 0.9], [0.4, 0., 0.7], [0.9, 0.7, 0.]])
        condensed = squareform(distances)
        average = fcluster(linkage(condensed, method='average'), 0.85, criterion='distance')
        complete = fcluster(linkage(condensed, method='complete'), 0.85, criterion='distance')
        assert len(np.unique(average)) == 1 and len(np.unique(complete)) == 2
        for group in np.unique(complete):
            rows = np.flatnonzero(complete == group)
            assert distances[np.ix_(rows, rows)].max() <= 0.85
        probabilities = np.full((24, 18), 1 / 18)
        groups = np.repeat([1, 2], 12)
        decoded = HungarianDecoder(True).decode(probabilities, groups, np.repeat([0.80, 0.90], 12))
        assert len(np.unique(decoded[:12].argmax(1))) == 12
        np.testing.assert_array_equal(decoded[12:], probabilities[12:])

GroupingContractChecks.run()


In [ ]:
'''10. Submission validation and experiment orchestration'''

In [ ]:
class SubmissionWriter:
    def __init__(self, config):
        self.config = config

    def write(self, test, probabilities, name='submission.parquet'):
        if probabilities.shape != (len(test), len(self.config.labels)) or not np.isfinite(probabilities).all():
            raise ValueError('Invalid submission prediction matrix')
        predictions = np.asarray(self.config.labels)[probabilities.argmax(1)]
        submission = pd.DataFrame({'id': test['id'].astype(str), 'model': predictions})
        if submission.isna().any().any() or submission['id'].duplicated().any():
            raise ValueError('Invalid submission IDs or labels')
        if not submission['model'].isin(self.config.labels).all():
            raise ValueError('Unknown submission label')
        submission.to_parquet(self.config.output_root / name, index=False)
        return submission

In [ ]:
class TwoStageExperiment:
    def __init__(self, config, gaussian_source=None):
        self.config = config
        self.assets = CompetitionAssets(config)
        self.extractor = LoRAFeatureExtractor(str(config.input_root), config.labels)
        self.gaussian_stack = None
        self.member_cache = {}

    def score_loras(self, train, test, targets, groups, eligible, logits):
        for name, settings in self.config.loras.items():
            directory = os.path.dirname(self.assets.find(f'{name}/meta.json'))
            base = self.extractor.find_base(settings['base'])
            with self.timer.measure(f'{name}: feature extraction'):
                texts = test['text'].tolist() + train['text'].astype(str).tolist()
                all_features, heads = self.extractor.lora_features(directory, base, texts, settings['bs'])
                features, training = all_features[:len(test)], all_features[len(test):]
            with self.timer.measure(f'{name}: Gaussian fit and group inference'):
                parameters = SharedPromptGaussian.fit(training, targets, train['prompt_id'].to_numpy(),
                    dict(red='wpca', d=128, se_shrink='lw', sp_shrink='lw', lam_floor=1e-4,
                         n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))
                parameters['T1'] = parameters['Tg'] = 3.0
                group_scores = np.full((len(test), len(self.config.labels)), np.nan)
                if eligible.any():
                    group_scores[eligible] = SharedPromptGaussian.scores(features[eligible],
                        np.unique(groups[eligible], return_inverse=True)[1], parameters)
                    if not np.isfinite(group_scores[eligible]).all():
                        raise ValueError('Non-finite Gaussian group probabilities')
                    logits[eligible] += (0.10 if name == 'lora8b_v1' else 0.15) * group_scores[eligible]
                rows = np.flatnonzero(train['prompt_id'].to_numpy() == train['prompt_id'].iloc[0])[:2]
                if len(rows) != 2:
                    raise ValueError('Need two training examples for the Gaussian pair path check')
                checked = SharedPromptGaussian.scores(training[rows], np.zeros(2, dtype=int), parameters)
                assert np.isfinite(checked).all() and np.allclose(np.exp(checked).sum(1), 1)
            with self.timer.measure(f'{name}: readouts and cache'):
                plain = ProbabilityMath.softmax(features @ heads['plain.weight'].T + heads['plain.bias'])
                logits += settings['w'] * np.log(np.clip(plain, 1e-9, 1))
                twin = np.full_like(plain, np.nan)
                if eligible.any():
                    centered = features.copy()
                    for group in np.unique(groups[eligible]):
                        centered[groups == group] -= centered[groups == group].mean(0)
                    twin[eligible] = ProbabilityMath.softmax(centered[eligible] @ heads['centered.weight'].T + heads['centered.bias'])
                    logits[eligible] += settings['twin_w'] * np.log(np.clip(twin[eligible], 1e-9, 1))
                    del centered
                self.member_cache[f'{name}_plain'] = plain
                self.member_cache[f'{name}_twin'] = twin
                self.member_cache[f'{name}_gaussian_log'] = group_scores
                np.savez_compressed(self.config.output_root / f'{name}_member_predictions.npz',
                    ids=test['id'].astype(str).to_numpy(dtype=str), labels=np.asarray(self.config.labels, dtype=str),
                    plain=plain, twin=twin, gaussian_log=group_scores, eligible=eligible)
                np.save(self.config.output_root / f'{name}_training_features.npy', training, allow_pickle=False)
                np.savez_compressed(self.config.output_root / f'{name}_heads.npz', **heads)
            del all_features, features, training, heads, parameters, texts, checked
            gc.collect()
            torch.cuda.empty_cache()
        return logits

    def run(self):
        self.config.output_root.mkdir(parents=True, exist_ok=True)
        output = self.config.output_root / 'submission.parquet'
        if output.exists():
            raise FileExistsError('Start a clean session; submission.parquet already exists')
        self.timer = StageTimer(self.config.output_root)
        started = time.monotonic()
        with self.timer.measure('input checks'):
            self.assets.preflight()
            train, test, targets = self.assets.load()
            for settings in self.config.loras.values():
                self.extractor.find_base(settings['base'])
        previous_directory = Path.cwd()
        os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
        try:
            os.chdir(self.config.output_root)
            with threadpool_limits(limits=4):
                return self.predict(train, test, targets, output, started)
        except BaseException:
            if output.exists():
                output.rename(output.with_name(f'failed_submission_{time.time_ns()}.parquet'))
            raise
        finally:
            os.chdir(previous_directory)

    def predict(self, train, test, targets, output, started):
        with self.timer.measure('style classifier'):
            pool = pd.read_parquet(self.assets.find('pool_v1.parquet'))
            probabilities = StyleClassifier(self.config.labels).fit_predict(train, test, targets, pool)
            del pool
            self.member_cache['svc'] = probabilities
            np.savez_compressed(self.config.output_root / 'svc_predictions.npz',
                ids=test['id'].astype(str).to_numpy(dtype=str), labels=np.asarray(self.config.labels, dtype=str),
                probabilities=probabilities)
            logits = W_SVC * np.log(np.clip(probabilities, 1e-9, 1))
        with self.timer.measure(f'topic grouping: {self.config.linkage_method}'):
            groups, eligible, maximum_distance = TopicGrouping(
                self.config.cluster_threshold, self.config.linkage_method).fit_predict(test['text'])
            print(f'Groups: {len(np.unique(groups))}; eligible rows: {int(eligible.sum())}', flush=True)
        logits = self.score_loras(train, test, targets, groups, eligible, logits)
        new_probability = ProbabilityMath.softmax(logits.copy())
        mixture = new_probability
        with self.timer.measure('final calibration, matching and artifacts'):
            if not np.isfinite(mixture).all() or not np.allclose(mixture.sum(1), 1):
                raise ValueError('Invalid ensemble probabilities')
            balanced = ProbabilityMath.balance(np.log(np.clip(mixture, 1e-12, 1)))
            mask = np.array([' — ' in text for text in test['text']], dtype=bool)
            before_hungarian = ProbabilityMath.punctuation_guard(balanced, mask, self.config.labels)
            decoder = HungarianDecoder(False, self.config.cluster_threshold)
            selected = decoder.decode(before_hungarian, groups, maximum_distance)
            np.savez_compressed(self.config.output_root / 'decoder_inputs.npz',
                ids=test['id'].astype(str).to_numpy(dtype=str), labels=np.asarray(self.config.labels, dtype=str),
                after_sinkhorn=balanced, before_hungarian=before_hungarian, groups=groups,
                max_pair_distance=maximum_distance, spaced_dash_mask=mask,
                source_sha256=np.array(SOURCE_SHA256))
            np.savez_compressed(self.config.output_root / 'ensemble_members.npz',
                ids=test['id'].astype(str).to_numpy(dtype=str), labels=np.asarray(self.config.labels, dtype=str),
                eligible=eligible, **self.member_cache)
            train[['id', 'prompt_id', 'model']].to_parquet(self.config.output_root / 'training_feature_rows.parquet', index=False)
            submission = SubmissionWriter(self.config).write(test, selected)
            report = {'mode': self.config.mode, 'original_author': 'Sangam Gautam (sangam21)',
                      'source_url': SOURCE_URL, 'source_sha256': SOURCE_SHA256,
                      'baseline_score': 0.604, 'baseline_score_evidence': 'reported by the user for iteration 07',
                      'this_iteration_score': None, 'use_complement': self.config.use_complement,
                      'checkpoint_discovery': 'Skip config-only directories and require all indexed weight shards',
                      'linkage_method': self.config.linkage_method, 'cluster_threshold': self.config.cluster_threshold,
                      'group_count': int(len(np.unique(groups))), 'eligible_rows': int(eligible.sum()),
                      'decoder': decoder.audit, 'test_rows': len(test),
                      'elapsed_seconds': time.monotonic() - started,
                      'cpu_blas_threads': 4, 'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                      'training_features_are_oof': False,
                      'cache_scope': 'Only the IDs and test pool used in this run; never load a visible cache during hidden scoring',
                      'submission_sha256': hashlib.sha256(output.read_bytes()).hexdigest()}
            (self.config.output_root / 'experiment_audit.json').write_text(json.dumps(report, indent=2))
            print(json.dumps({key: report[key] for key in ['mode', 'test_rows', 'elapsed_seconds']}, indent=2), flush=True)
        return submission


In [ ]:
'''11. Run and inspect artifacts
Save Version / Run All in a fresh GPU T4 x2 session; submit only a completed version.
No timed fallback silently drops models or changes the selected experiment.
stage_timings.json is updated at each stage start/end; upstream stack logs show its inner progress.
The experiment writes /kaggle/working/submission.parquet.
The inference cache belongs only to this run and is never automatically reused during scoring.
A three-row visible cache cannot replace a hidden-test rerun.
Training feature caches use official row order, described by training_feature_rows.parquet.
These are fixed pretrained features, not out-of-fold predictions: LoRA checkpoints may have seen those rows.
Future Gaussian/readout validation must split by prompt_id and fit preprocessing and heads within each fold.
Feature export adds disk I/O and requires output storage. No target runtime or score is guaranteed.'''

In [ ]:
config = ExperimentConfig()
experiment = TwoStageExperiment(config)
submission = experiment.run()
submission.head()